# P24 run notebook (Kaggle or Colab, GPU on)

Pick **one JOB** per notebook session. Run all cells. Results are logged after every single run, so if the session dies, run all cells again and it continues where it stopped.

| JOB | Who | What | Rough GPU time |
|---|---|---|---|
| `A` | Tirthkumar (Kaggle) | ETTh1 reproduction, 4 models x 4 horizons x 3 seeds, plus F1 re-scoring | a few hours; time the first runs |
| `B` | Sneh (Kaggle) | Build ON-Zonal from IESO, then RQ2 preliminary (real vs shifted zones, 3 seeds) | a few hours |
| `C` | Sneh (Colab) | PatchTST patch-length and look-back ablation on ETTh1 | 1-2 h per seed |
| `D` | Tirthkumar (Colab), only if time | ETTm1 reproduction, 1 seed | several hours |
| `E` | Sneh (Kaggle), Week 7 | RQ2 timestamp check: rebuild ON-Zonal, then iTransformer without calendar tokens on real and shifted zones (6 runs) | about 2.5 h |

**Kaggle:** Settings → Accelerator: GPU; Internet: On (needs a phone-verified account). Then *Save Version → Save & Run All* so it keeps running after you close the tab. Download the `results_*` and `logs_*` folders from the Output tab when done.

**Colab:** Runtime → Change runtime type → T4 GPU. Results go to your Google Drive so they survive a disconnect. Keep the tab open.

In [ ]:
# ---- choose the job ----
JOB = "A"          # "A", "B", "C", "D" or "E"
SEEDS = [2021, 2022, 2023]   # for JOB C/D you may start with [2021] only
KIT_REPO = "https://github.com/Tim64465/CISC-473-Project"   # your repo, after you push the kit

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    BASE = "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/P24"
os.makedirs(BASE, exist_ok=True)
os.environ["P24_RESULTS"] = f"{BASE}/results_{JOB}"
os.environ["P24_LOGS"] = f"{BASE}/logs_{JOB}"
os.environ["P24_WORK"] = "/tmp/p24_work"     # repos and checkpoints (fast local disk)
os.environ["P24_TABLES"] = f"{BASE}/tables_{JOB}"
print("results ->", os.environ["P24_RESULTS"])

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install reformer_pytorch
import torch, numpy, pandas; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| numpy", numpy.__version__, "| pandas", pandas.__version__)
assert torch.cuda.is_available(), "No GPU: turn the GPU accelerator on, then restart"


In [ ]:
%cd /tmp
!rm -rf kit && git clone -q $KIT_REPO kit
%cd /tmp/kit
!python p24.py setup

In [ ]:
seeds = " ".join(map(str, SEEDS))
if JOB == "A":
    !python p24.py run --plan A --seeds {seeds}
elif JOB == "B":
    !python onzonal.py
    !python p24.py run --plan onzonal --seeds {seeds}
elif JOB == "C":
    !python p24.py run --plan ablation --seeds {seeds}
elif JOB == "D":
    !python p24.py run --plan ETTm1 --seeds {seeds}
elif JOB == "E":
    !python onzonal.py
    !python p24.py run --plan onzonal_nots --seeds {seeds}

In [ ]:
# Tables from this job's runs (merge all jobs later on one machine: python tables.py results)
!python tables.py {os.environ['P24_RESULTS']}
!cd {BASE} && zip -qr results_{JOB}.zip results_{JOB} logs_{JOB} && ls -la {BASE}